## Lab 1 (Weeks 2–4): Opportunity Line Item Dimensional Model

> Before starting, review these resources:
> - `lab1_intro.md` — concepts covered in lecture

### Learning Objectives
By the end of this lab you will be able to:
- Declare the grain of a business process
- Design dimension and fact tables using a star schema
- Apply a **Type 1 SCD** strategy
- Load data from an OLTP source into a dimensional model

### Business Scenario
Sales leadership wants to analyze **product-level revenue** by customer and product across sales opportunities. Source data is in `SNOWBEARAIR_DB.PUBLIC`. You will build your dimensional model in the `MODELED` schema.

---
### Naming Convention
All table names must end with your `_LAST_FI` suffix — your last name and first initial, lowercase.

**Example:** if your name is Jane Clark → `_clark_j`

Save this notebook as `LAST_FI_LAB1` (e.g., `CLARK_J_LAB1`) in the `MODELED` schema.

---
## Setup
Run the cell below to point your session at the correct database and schema. **Update the `suffix` variable with your own `_LAST_FI` value before running anything else.**

In [ ]:
USE DATABASE SNOWBEARAIR_DB;
USE SCHEMA MODELED;
USE WAREHOUSE RAW_WH;

---
## Explore the Source Tables
Before designing anything, run these queries to understand what data you are working with.
Pay attention to column names, data types, and how the tables relate to each other.

In [ ]:
-- Explore each source table — run these one at a time
SELECT * FROM SNOWBEARAIR_DB.PUBLIC.product                LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.account             LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.opportunity         LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.opportunity_line_item LIMIT 10;

---
## Task 1: Declare the Business Process and Grain

Before writing any SQL, answer the following in the markdown cell below:

1. What **business process** are we measuring?
2. What does **one row** in your fact table represent?

Write your grain statement in plain English: *"One row per..."*

> **Hint:** Use the Kimball 4-step process from the intro. Look at `opportunity_line_item` — what is the most granular event recorded there?

**Business Process:**
*(write your answer here)*

**Grain Statement:**
*(write your answer here — e.g., "One row per...")*

---
# Part 1: Design — Create Your Tables

Build your dimensions first, then the fact table. Foreign keys in the fact table must reference tables that already exist.

**Remember:** Append your `_LAST_FI` suffix to every table name.

---
### Task 2: Design the Product Dimension (Type 1)

Create `dim_product_last_fi`

**Requirements:**
- `product_key` — surrogate primary key (`AUTOINCREMENT`)
- `product_id` — natural key from the source system
- Descriptive attributes: name, product code, family, active flag
- **Type 1 SCD** — this table will be updated in place; no history is preserved

**Tip:** Check the column names in the source `product` table before writing your DDL.

In [ ]:
-- Task 2: Create dim_product_last_fi
-- Replace 'last_fi' with your own suffix throughout this notebook

CREATE OR REPLACE TABLE dim_product_last_fi (
    product_key     INTEGER AUTOINCREMENT PRIMARY KEY,
    product_id      VARCHAR(255)  NOT NULL,  -- natural key
    -- add your descriptive attributes below

);

---
### Task 3: Design the Account Dimension (Type 1)

Create `dim_account_last_fi`

**Requirements:**
- `account_key` — surrogate primary key (`AUTOINCREMENT`)
- `account_id` — natural key from the source system
- Descriptive attributes: name, type, industry, billing state, annual revenue, number of employees
- **Type 1 SCD**

In [ ]:
-- Task 3: Create dim_account_last_fi

CREATE OR REPLACE TABLE dim_account_last_fi (
    account_key     INTEGER AUTOINCREMENT PRIMARY KEY,
    account_id      VARCHAR(255)  NOT NULL,  -- natural key
    -- add your descriptive attributes below

);

---
### Task 4: Design the Opportunity Dimension (Type 1)

Create `dim_opportunity_last_fi`

**Requirements:**
- `opportunity_key` — surrogate primary key (`AUTOINCREMENT`)
- `opportunity_id` — natural key from the source system
- Descriptive attributes: name, stage, close date, type
- **Type 1 SCD**

**Note:** The `opportunity` table is the bridge between line items and accounts. Understanding its structure is essential for loading the fact table in Part 2.

In [ ]:
-- Task 4: Create dim_opportunity_last_fi

CREATE OR REPLACE TABLE dim_opportunity_last_fi (
    opportunity_key     INTEGER AUTOINCREMENT PRIMARY KEY,
    opportunity_id      VARCHAR(255)  NOT NULL,  -- natural key
    -- add your descriptive attributes below

);

---
### Task 5: Design the Fact Table

Create `fact_opportunity_line_item_last_fi`

**Requirements:**
- A surrogate primary key
- Foreign keys to `dim_product_last_fi`, `dim_account_last_fi`, and `dim_opportunity_last_fi`
- Measures: quantity, unit price, total price

**Think about data types:** Which measures are additive (safe to SUM)? Which are not? Choose your data types accordingly.

> **Reminder:** The fact table stores foreign keys (surrogate integers) — never the natural key strings from the source system.

In [ ]:
-- Task 5: Create fact_opportunity_line_item_last_fi

CREATE OR REPLACE TABLE fact_opportunity_line_item_last_fi (
    line_item_key       INTEGER AUTOINCREMENT PRIMARY KEY,
    -- foreign keys (surrogate integers only)

    -- measures

);

---
# Part 2: Load Data from the OLTP Source

Now populate your tables using data from `SNOWBEARAIR_DB.PUBLIC`.

Use `INSERT INTO ... SELECT ...` to load each dimension table, then load the fact table last.

**Key rule:** Always load dimensions before the fact table. The fact table JOIN needs the surrogate keys that dimensions generate.

---
### Load dim_product

Insert from `SNOWBEARAIR_DB.PUBLIC.product` into your dimension.
Select the columns that match your dimension table definition.

In [ ]:
-- Load dim_product_last_fi

INSERT INTO dim_product_last_fi (
    -- your columns
)
SELECT
    -- your source columns
FROM SNOWBEARAIR_DB.PUBLIC.product;

---
### Load dim_account

Insert from `SNOWBEARAIR_DB.PUBLIC.account` into your dimension.

In [ ]:
-- Load dim_account_last_fi

INSERT INTO dim_account_last_fi (
    -- your columns
)
SELECT
    -- your source columns
FROM SNOWBEARAIR_DB.PUBLIC.account;

---
### Load dim_opportunity

Insert from `SNOWBEARAIR_DB.PUBLIC.opportunity` into your dimension.

In [ ]:
-- Load dim_opportunity_last_fi

INSERT INTO dim_opportunity_last_fi (
    -- your columns
)
SELECT
    -- your source columns
FROM SNOWBEARAIR_DB.PUBLIC.opportunity;

---
### Load fact_opportunity_line_item

Insert from `SNOWBEARAIR_DB.PUBLIC.opportunity_line_item`.

**You must resolve surrogate keys by joining through your dimension tables** — do not load the raw string IDs from the source into the fact table.

`opportunity_line_item` does not have a direct `account_id`. You will need to join through `opportunity` to get it.

**Join path:**
```
opportunity_line_item
  → opportunity        (to get account_id)
  → dim_account        (to get account_key)
  → dim_product        (to get product_key)
  → dim_opportunity    (to get opportunity_key)
```

In [ ]:
-- Load fact_opportunity_line_item_last_fi
-- Join through dimension tables to resolve surrogate keys

INSERT INTO fact_opportunity_line_item_last_fi (
    -- your foreign key columns
    -- your measure columns
)
SELECT
    -- surrogate keys from your dimension tables
    -- measures from oli
FROM SNOWBEARAIR_DB.PUBLIC.opportunity_line_item oli
-- add your JOINs here following the join path above
;

---
## Verify Your Work

Run the queries below to confirm your tables loaded correctly before submitting.

- Do the row counts look right?
- Can you join the fact table back to all three dimensions?
- Try a simple aggregation: total revenue by product family.

In [ ]:
-- Row counts
SELECT 'dim_product'               AS table_name, COUNT(*) AS row_count FROM dim_product_last_fi
UNION ALL
SELECT 'dim_account',                              COUNT(*)              FROM dim_account_last_fi
UNION ALL
SELECT 'dim_opportunity',                          COUNT(*)              FROM dim_opportunity_last_fi
UNION ALL
SELECT 'fact_opportunity_line_item',               COUNT(*)              FROM fact_opportunity_line_item_last_fi;

-- Sanity check: total revenue by product family
-- SELECT
--     p.product_family,
--     SUM(f.total_price) AS total_revenue
-- FROM fact_opportunity_line_item_last_fi f
-- JOIN dim_product_last_fi p ON p.product_key = f.product_key
-- GROUP BY p.product_family
-- ORDER BY total_revenue DESC;

---
## Deliverables Checklist

Before submitting on Canvas:

- [ ] Notebook saved as `LAST_FI_LAB1` in the `MODELED` schema
- [ ] All table names include your `_LAST_FI` suffix
- [ ] Task 1 grain answer written in the markdown cell above
- [ ] DDL cells run successfully for all 4 tables (Tasks 2–5)
- [ ] All 4 load cells run successfully with matching row counts
- [ ] Star schema diagram uploaded **or** DDL written in full (either/or)
- [ ] Short write-up submitted in the Canvas text box covering:
  - Your grain choice and reasoning
  - Why Type 1 SCD is appropriate for each dimension in this lab